
# Laboratorio 5 — Train-Test Split y Validación Cruzada
## Versión **estudiante**

**Curso:** Machine Learning (EIN143A25) · Paralelo 701
**Fecha:** Miércoles 2 de septiembre de 2026 · Sesión 5 de 17
**Bloques:** 6-8 (11:40-13:40) · Lab. Central 114
**Unidad 1:** Fundamentos y Preparación (cierre)

**Objetivo:** ver en código, con números propios, todo lo que la teoría explicó hoy — la varianza
de un solo split, la curva de sobre/subajuste variando la complejidad de KNN, y cómo la
validación cruzada da una estimación más estable. Cierre de la Unidad 1: el pipeline completo,
de punta a punta, sobre Wine.

> **De dónde venimos:** en el Lab 4 usamos `random_state=42` "de prestado" en cada split, sin
> preguntarnos qué pasaría con otro número. Hoy lo respondemos — y de paso cerramos la Unidad 1
> completa: selección (Lab 4), extracción (Lab 4), escalamiento (Lab 3), y ahora cómo evaluar todo
> eso sin engañarse.


---
## Parte 1 · La grieta del split simple, en código


### 1. Repetir el mismo split con semillas distintas

Un solo `train_test_split` da un solo número. ¿Ese número es "el" score del modelo, o solo le
tocó una partición con suerte (buena o mala)?

**Antes de correr nada, responde:** ¿esperas que el score sea siempre el mismo número, que cambie
mucho, o que cambie poco entre semillas?

> *(tu respuesta)*

**Tu tarea:**
1. Para `seed` en `range(10)`: parte Wine con `train_test_split(..., test_size=0.3,
   random_state=seed, stratify=y)`, escala (`fit` en train, `transform` en test) y entrena un
   `KNeighborsClassifier()` por defecto.
2. Guarda cada score de test en una lista.
3. Imprime la lista completa y su media.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

wine = load_wine(as_frame=True)
X, y = wine.data, wine.target


**Responde aquí:** mirando los 10 scores, ¿varían poco o mucho para un dataset de solo 178
muestras? ¿Qué significaría reportar *un solo* score de *un solo* split como "el" desempeño del
modelo?

> *(tu respuesta)*


### 2. Estratificación en acción

**Tu tarea:**
1. Con un split (`random_state=42, stratify=y`), compara `y_train.value_counts(normalize=True)`
   contra `y.value_counts(normalize=True)`.
2. Repite el split **sin** `stratify=y` y vuelve a comparar las proporciones de `y_train`.

**Responde aquí:** ¿qué tan parecidas quedan las proporciones con `stratify=y` frente a sin él?
Con un dataset chico como Wine (178 filas, 3 clases), ¿por qué importa más este detalle que en un
dataset de un millón de filas?

> *(tu respuesta)*


### 3. Sobreajuste y subajuste con KNN: la curva en U

**Tu tarea:**
1. Parte Wine una vez (`random_state=42, stratify=y`) y escala.
2. Para `k` en `range(1, 31)`: entrena un `KNeighborsClassifier(n_neighbors=k)` y guarda **tanto**
   el score de train como el de test.
3. Grafica ambas curvas contra `k`, con leyenda.

In [ ]:
import matplotlib.pyplot as plt


**Responde aquí:**
1. ¿Dónde está train más alto que test — y por qué eso es la firma del sobreajuste?
2. ¿La curva de test es una U perfecta y suave, o tiene ruido? Si el "mejor k" de tu gráfico
   fuera 1 o 2, ¿confiarías en elegirlo así, con un test set de solo ~54 filas?
3. Conecta con la teoría de esta mañana: ¿por qué agrandar `k` reduce la varianza del modelo pero
   aumenta su sesgo?

> *(tus respuestas)*

---
## ☕ Recreo

---
## Parte 2 · Validación cruzada


### 4. `cross_val_score`, la herramienta

**Tu tarea:**
1. Arma un `Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=7))])`.
2. Pásalo a `cross_val_score(pipe, X, y, cv=5)` — **sobre todo `X`, `y`, sin partir antes.**
3. Imprime los 5 scores y su media ± desviación estándar.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline


**Responde aquí:** ¿por qué se usa un `Pipeline` acá en vez de escalar `X` una sola vez con
`fit_transform` antes de llamar a `cross_val_score`? *(pista: `cross_val_score` entrena y evalúa
5 veces, con particiones distintas cada vez — ¿qué le pasaría a la regla de oro de la Clase 3 si
el escalamiento se hiciera una sola vez, afuera, sobre todos los datos?)* Compara la media ±
desviación de acá contra los 10 scores individuales de la sección 1: ¿es más estable?

> *(tu respuesta)*


### 5. Validación cruzada estratificada

**Tu tarea:** crea un `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` y pásalo como
`cv=` a `cross_val_score` con el mismo pipeline de la sección anterior. Compara media y desviación
contra el `cv=5` simple.

In [ ]:
from sklearn.model_selection import StratifiedKFold


**Responde aquí:** ¿cambió mucho el resultado frente al `cv=5` simple? En Wine, más o menos
balanceado, ¿esperabas una diferencia grande o chica? ¿En qué tipo de dataset la estratificación
en CV importaría mucho más?

> *(tu respuesta)*


### 6. Cross-validation para elegir k — cerrando el círculo

**Tu tarea:**
1. Para `k` en `range(1, 31)`: arma un pipeline (escalar + KNN con ese `k`) y calcula
   `cross_val_score(..., cv=5).mean()`.
2. Grafica la curva de score promedio de CV contra `k`.
3. Encuentra el `k` con mejor score promedio y guárdalo en `best_k`.

In [ ]:
import numpy as np


**Responde aquí:** compara el `best_k` de esta curva contra el "mejor k" que se veía en el
gráfico de la sección 3 (un solo split) — ¿coinciden? Si no coinciden, ¿en cuál de los dos confías
más, y por qué?

> *(tu respuesta)*

---
## Parte 3 · Pipeline completo y cierre de la Unidad 1


### 7. El pipeline de punta a punta, en una sola celda

Síntesis de toda la Unidad 1, sobre Wine:

**Tu tarea:**
1. Parte Wine (`test_size=0.3, random_state=42, stratify=y`).
2. Arma un `Pipeline` con tres pasos: `SelectKBest(score_func=f_classif, k=8)` (Clase 4),
   `StandardScaler()` (Clase 3), `KNeighborsClassifier(n_neighbors=best_k)` (el `k` elegido por CV
   en la sección 6).
3. Corre `cross_val_score` de ese pipeline **sobre el train** (`cv=5`) e imprime media ± desviación.
4. Ajusta el pipeline completo en train y evalúalo en test — el test set que **nunca** se tocó
   hasta esta línea.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif


**Responde aquí:** recorre las tres líneas del pipeline y nombra de qué clase viene cada paso.
¿En qué momento, en todo este proceso, se usó el test set para decidir algo — ajustar un
escalador, elegir features, elegir `k`? Si la respuesta es "nunca", ¿por qué eso importa?

> *(tu respuesta)*


### 8. Ejercicio propuesto (no evaluado)

Repite el pipeline completo de la sección 7, pero cambia `KNeighborsClassifier` por
`DecisionTreeClassifier` y en vez de `n_neighbors` barre `max_depth` (por ejemplo
`range(1, 15)`) con CV, igual que hiciste con `k` en la sección 6.

¿La curva de score promedio vs `max_depth` muestra la misma forma de U que la de `k`? ¿Por qué un
árbol muy profundo sobreajustaría, en paralelo a un KNN con `k` muy chico?

In [ ]:
from sklearn.tree import DecisionTreeClassifier



---
**Moraleja del lab — y de la Unidad 1 completa:** cada duda del Lab 1 ("¿son creíbles estos
scores?") se convirtió, clase a clase, en una herramienta concreta: escalar bien (Clase 3), sacar
columnas de sobra (Clase 4), y ahora evaluar sin engañarse (Clase 5). El pipeline de la sección 7
es el resumen ejecutable de las cinco clases.

**Anuncios:**
- **Certamen 1: miércoles 9 de septiembre.** Cubre las cinco sesiones completas de la Unidad 1.
- La **Tarea 1** (clases 1 a 5) se enunciará antes del certamen.
- Después del certamen arranca la **Unidad 2 — Algoritmos supervisados y no supervisados**,
  empezando por regresión lineal y logística.